# MoE / SER Expert Routing

Route tokens to the top-2 of 8 experts with `SERModel`, compute the
load-balancing loss, and visualize the routing distribution. The routing
math is reproduced in pure NumPy for the no-C-backend case.

In [ ]:
import numpy as np
from SneppX_ALG import SERModel, SERConfig, Tensor
import SneppX_ALG as S
HAS_C = S._HAS_C_BACKEND
print('C backend:', HAS_C)

## Configure an 8-expert, top-2 MoE

In [ ]:
cfg = SERConfig()
cfg.num_experts = 8; cfg.num_active = 2
cfg.input_dim = 64; cfg.expert_dim = 128; cfg.output_dim = 64
cfg.top_k_method = 0; cfg.load_balance_coef = 0.01; cfg.dropout_rate = 0.0
moe = SERModel(cfg, seed=42, num_layers=1)
print('experts:', cfg.num_experts, 'active:', cfg.num_active)

## Forward

In [ ]:
x = Tensor.randn((4, 16, cfg.input_dim))
if HAS_C:
    out = moe.forward(x)
    print('forward out:', out.shape)
else:
    print('C backend required for SERModel.forward; see NumPy router below')

## NumPy routing illustration (no C backend needed)

In [ ]:
rng = np.random.default_rng(0)
router_logits = rng.standard_normal((16, 8))      # (seq, experts)
scores = np.exp(router_logits - router_logits.max(-1, keepdims=True))
scores /= scores.sum(-1, keepdims=True)
top2 = np.argsort(-scores, axis=-1)[:, :2]
load = np.zeros(8)
for t in range(16):
    load[top2[t]] += 1
print('tokens-per-expert:', load.astype(int))

## Load-balancing loss

In [ ]:
frac = load / 16.0
p_avg = scores.mean(axis=0)
loss = float(np.sum(frac * p_avg) * cfg.load_balance_coef)
print('balance loss:', round(loss, 6))

## Stacked MoE layer

In [ ]:
from SneppX_ALG import Module, Linear
class MoETransformer(Module):
    def __init__(self):
        super().__init__()
        self.inp = Linear(64, 64)
        self.moe = SERModel(SERConfig(), seed=1, num_layers=4)
        self.out = Linear(64, 64)
    def forward(self, x):
        return self.out(self.moe.forward(self.inp(x)))

net = MoETransformer()
print('MoETransformer ready')